In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 03 \u2014 Responsibilities"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Objectives\n",
        "- Compute posterior component probabilities manually.\n",
        "- Contrast soft and hard assignments.\n",
        "- Visualize confidence across the input plane."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Mathematical Background\n",
        "$\\gamma_{ik}=p(z_i=k\\mid x_i)=\\frac{\\pi_k\\mathcal N(x_i\\mid\\mu_k,\\Sigma_k)}{\\sum_j\\pi_j\\mathcal N(x_i\\mid\\mu_j,\\Sigma_j)}$."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Implementation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import sys\n",
        "from pathlib import Path\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "PROJECT = Path.cwd()\n",
        "if PROJECT.name == 'notebooks':\n",
        "    PROJECT = PROJECT.parent\n",
        "FIGURES = PROJECT / 'figures'\n",
        "FIGURES.mkdir(exist_ok=True)\n",
        "if str(PROJECT) not in sys.path:\n",
        "    sys.path.insert(0, str(PROJECT))\n",
        "\n",
        "plt.style.use('seaborn-v0_8-whitegrid')\n",
        "COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']\n",
        "RNG = np.random.default_rng(7)\n",
        "\n",
        "def savefig(name):\n",
        "    plt.tight_layout()\n",
        "    plt.savefig(FIGURES / name, dpi=180, bbox_inches='tight', transparent=True)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Experiments"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from src.gaussian import sample_mixture\n",
        "from src.em import e_step\n",
        "weights = np.array([.45,.35,.20]); means = np.array([[-2,0],[1.5,1.2],[2.2,-1.4]])\n",
        "covs = np.array([[[.8,.1],[.1,.5]], [[.7,.3],[.3,.7]], [[.45,-.1],[-.1,.5]]])\n",
        "X, z = sample_mixture(weights, means, covs, 900, random_state=RNG)\n",
        "gamma, _ = e_step(X, weights, means, covs)\n",
        "conf = gamma.max(axis=1)\n",
        "plt.figure(figsize=(6,5))\n",
        "plt.scatter(X[:,0], X[:,1], c=gamma[:,0], cmap='viridis', s=18, alpha=.75)\n",
        "plt.colorbar(label='responsibility for component 0')\n",
        "plt.title('Where is component 0 responsible?'); savefig('responsibilities.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "xx, yy = np.meshgrid(np.linspace(-5,5,220), np.linspace(-4,4,220))\n",
        "grid = np.column_stack([xx.ravel(), yy.ravel()])\n",
        "grid_gamma, _ = e_step(grid, weights, means, covs)\n",
        "plt.figure(figsize=(7,5))\n",
        "plt.contourf(xx, yy, grid_gamma.max(axis=1).reshape(xx.shape), levels=20, cmap='magma')\n",
        "plt.scatter(X[:,0], X[:,1], s=8, color='white', alpha=.35)\n",
        "plt.colorbar(label='max posterior probability')\n",
        "plt.title('Where is the hard decision confident?'); savefig('heatmap.png')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Observations\n",
        "- Posterior probabilities change smoothly near overlaps.\n",
        "- Hard labels discard uncertainty that is visible in the heatmap."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Key Takeaways\n",
        "- Responsibilities are Bayes-rule weights.\n",
        "- They are the sufficient bridge from latent variables to EM updates."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Transition to the next notebook\n",
        "Next we alternate responsibility computation with parameter updates: the EM algorithm."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "pygments_lexer": "ipython3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}